# Tutorial 1 — Quick start

This tutorial walks through the minimal kaleidocell workflow from raw AnnData to a
self-contained HTML report.  It covers:

1. Running NMF across all samples simultaneously
2. Clustering NMF programs into consensus meta-programs (MPs)
3. Visualising the similarity matrix
4. Scoring cells by MP activity
5. Generating a full HTML report

**Dataset** — Peter Sims lab DMSO vs. panobinostat scRNA-seq cohort
(5 patients, ~15 000 cells, log-normalised counts stored in `adata.X`).

## 0 · Imports

In [ ]:
import kaleidocell
import scanpy as sc

print(f"kaleidocell version: {kaleidocell.__version__}")

## 1 · Load data

In [ ]:
adata = sc.read_h5ad('../data/petersims_example.h5ad')
print(adata)

## 2 · Run NMF on every sample

`multi_sample_nmf` loops over each unique value in `batch_key`, extracts the
corresponding cells, and runs NMF at each rank in `test_ranks` with
`n_initializations` random starts.  The factorisation with the lowest
Frobenius error is retained per rank per sample.

**Key parameters**
| Parameter | Description |
|-----------|-------------|
| `batch_key` | `adata.obs` column identifying samples |
| `test_ranks` | range of NMF ranks to evaluate |
| `n_initializations` | random restarts per rank (higher = more stable, slower) |
| `max_iterations` | maximum multiplicative-update steps per init |

In [ ]:
results_nmf, nmf_convergence = kaleidocell.multi_sample_nmf(
    adata,
    batch_key='Patients',
    test_ranks=[4, 5, 6, 7, 8, 9], # default
    n_initializations=1, # default
    max_iterations = 100  # default
)

## 3 · Derive consensus meta-programs

All W matrices from all samples and ranks are concatenated.  Programs are
clustered by cosine similarity using Ward linkage.

`n_meta_programs` is automatically chosen when not specified (see
`kaleidocell.find_optimal_n_mp`).

In [ ]:
results_mp = kaleidocell.derive_nmf_metaprograms(
    results_nmf
)

## 4 · Inspect the similarity heatmap

The heatmap shows pairwise cosine similarities between all NMF programs,
sorted by cluster assignment.  Tight diagonal blocks indicate well-separated
meta-programs.

In [ ]:
kaleidocell.plot_heatmap(results_mp)

## 5 · Inspect quality metrics

| Metric | Meaning |
|--------|---------|
| `sampleCoverage` | Fraction of samples contributing at least one program to this MP |
| `silhouette` | Within-cluster cohesion (−1 to +1; > 0.2 is good) |
| `meanSimilarity` | Average pairwise cosine similarity inside the cluster |
| `nPrograms` | Number of NMF programs assigned to this MP |
| `nGenes` | Size of the consensus gene signature |

In [ ]:
results_mp['metrics']

## 6 · Score cells by MP activity

`compute_mp_scores` projects each cell onto each MP by computing the mean
expression of the MP's top genes.  The result is a `(n_cells × n_MPs)`
DataFrame.

In [ ]:
mp_scores = kaleidocell.compute_mp_scores(results_mp, adata)
mp_scores.head()

## 7 · Generate HTML report

`get_html` produces a self-contained tabbed HTML file with all results
embedded as base64 images — no external dependencies needed to open it.
See *Output files* in the documentation for the full list of files written
alongside the HTML.

In [ ]:
path = kaleidocell.get_html(
    results_mp,
    adata,
    mp_scores=mp_scores,
    obs=['Treatment', 'Patients'],  # violin-plot tabs
    output_path='../results/01/', # Will be created if non existant
)
print(f"Report written to: {path}")